# Practical session 2: Implementing reactive behaviors

**Reminder:** 
- Each time you encounter a cell containing code in this notebook (as in the cell starting with `from vivarium.controllers...` below), you have to execute it by clicking on the cell and pressing `Shift+Enter` (unless it is explicitly specified not to do it). 

Let's connect this notebook to the simulator for this new session:

In [27]:
from vivarium.controllers import VivariumController
controller = VivariumController.start_session(scene_name="session_2")

INFO:vivarium.controllers.vivarium_controller:Server already running with scene 'session_2'. Connecting.
INFO:vivarium.controllers.vivarium_controller:Controller thread started on client
INFO:vivarium.controllers.vivarium_controller:Jupyter was started from Panel interface; skipping interface start. If you want to start another interface, set the environment variable VIVARIUM_JUPYTER_FROM_PANEL to 0.
INFO:vivarium.controllers.vivarium_controller:Controller thread is already running
INFO:vivarium.controllers.vivarium_controller:VivariumController session 'session_2' is started


We will use a single agent in this session (the blue square on the map), let's create an alias for it as we did in Session 1:

In [28]:
agent = controller.agents[0]

There is also a single object (the yellow circle), let's create an alias for it:

In [29]:
object = controller.objects[0]

And let place the agent and the object at specific positions and orientations, it will be useful for this session:

In [30]:
agent.x_position = 20
agent.y_position = 50
agent.orientation = 0
object.x_position = 50
object.y_position = 40

## Behaviors

In the last practical session we saw how to set the agent left and right motor activations as well as how to read the values returned by the left and right proximeters. We programmed a first simple behavior where the agent slows down when approaching an obstacle. 

Here is a possible solution for this behavior, if will run for approximately 10 seconds if you execute the next cell:

In [6]:
from time import sleep

# Repeat 100 times the indented code:
for i in range(100):
    # print the iteration number every 10 iterations
    if i % 10 == 0:
        print(f"Iteration {i}")
        
    # Read the proximeter values and store them in the "left" and "right" variables
    left, right = agent.proximeters()
    
    # Compute the maximum of the values returned by the left and right proximeters.
    max_prox_value = max(left, right)
    
    # Compute the activation that will be applied to both motors. 
    # The closer the obstacle (i.e. the higher the max value of the proximeters), the lower the motor activation should be.
    # Note that motor activation is bounded between 0 and 1
    motor_activation = (1.0 - max_prox_value)

    print(max_prox_value, motor_activation)
    
    # Set the activation of both motors to the value we just have computed
    agent.left_motor = motor_activation
    agent.right_motor = motor_activation
    
    # Waits for 100 milliseconds before starting the next iteration (to avoid overloading you computer)
    sleep(0.1)

# Stop the robot
print("Stop the agent's motors")
agent.stop_motors()

Iteration 0
0.19590747356414795 0.804092526435852
0.22362977266311646 0.7763702273368835
0.2875378131866455 0.7124621868133545
0.34103357791900635 0.6589664220809937
0.39233577251434326 0.6076642274856567
0.43918144702911377 0.5608185529708862
0.478099524974823 0.521900475025177
0.5156412720680237 0.4843587279319763
0.5468272566795349 0.4531727433204651
0.5800859928131104 0.41991400718688965
Iteration 10
0.610461413860321 0.38953858613967896
0.6369253396987915 0.3630746603012085
0.6611541509628296 0.3388458490371704
0.6812694668769836 0.31873053312301636
0.7017555236816406 0.2982444763183594
0.7214168310165405 0.2785831689834595
0.7401589751243591 0.25984102487564087
0.7550128698348999 0.2449871301651001
0.7686690092086792 0.2313309907913208
0.7806256413459778 0.21937435865402222
Iteration 20
0.7928174734115601 0.20718252658843994
0.8040088415145874 0.1959911584854126
0.8137983083724976 0.18620169162750244
0.8219335079193115 0.17806649208068848
0.8303104639053345 0.16968953609466553
0.

You should see the agent moving forward, slowing down while it approaches the object and re-accelerating when it passes it. Once the loop is completed it will print "Stop the agent's motor" and the agent will stop.

## Practical definition of a behavior

The example behavior defined above illustrates the general structure of a behavior. 

**Definition:** a behavior consists of a loop repeated at a certain frequency where: 
- (1) the values of relevant sensors are read,
- (2) some computation is performed using these values,
- (3) commands are sent to the agent motors according to the result of this computation.

In the example behavior above, step (1) corresponds to the reading of the left and right proximeters activations. Step (2) corresponds to the computation of `motor_activation` according to the maximum of the proximeter activations. Finally, Step (3) corresponds to setting the speed of both motors to the value of `motor_activation`.

Note that the code above will take some time to be executed (approximately `100 * 0.1 = 10` seconds, since the loop is repeated 100 times with a waiting time of 0.1 second at each iteration). During this time, you can't execute anything else in this notebook. To stop the execution before it terminates by itself you have manually stop the cell execution (by pressing the "stop-like" button, located either in the top menu bar of this document or next to the executing cell).  

This approach has three major drawbacks:
- Only one behavior can run at a time.
- The behavior has a fixed duration (at some point it will stop)
- We can't stop a behavior programmatically (instead we have to press the "stop-like" button).

To overcome these problems, we provide below a more flexible method for defining and executing behaviors. Let's rewrite the behavior above using that method, where defining a behavior boils down to defining a function which includes the core of the behavioral loop:

In [31]:
# The code in this cell defines a function called slow_down (first line),
# which takes as argument the agent (first line, in parenthesis),
# and returns the left and right wheel activation to be applied to the motors (last line)

def slow_down(agent):
    # Step (1): read the sensor values
    left, right = agent.proximeters()
    
    # Step (2): do some computation
    max_prox_value = max(left, right)
    motor_activation = 1.0 - max_prox_value
    
    # Step (3): return the motor activations for left and right motors (in this order)
    return motor_activation, motor_activation

The cell above defines a function called `slow_down`. In computer programming, a function is a sequence of instructions that executes a specific task depending on some parameters (called the arguments of the function) and that returns a result. In this sense it is very similar to the mathematical definition of a function, as for example when we write `y = f(x)`, where `f` is the name of the function, `x` is its argument, and `y` is the result. For example, we can define a function `square` that computes the square of its argument as follows:


In [8]:
def square(x):
    return x * x

print('The square of 3 is', square(3))

print('The square of 5 is', square(5))

The square of 3 is 9
The square of 5 is 25



As seen above, the definition of a function in Python starts with the keyword `def`, followed by an arbitrary name we choose for the function (above we called it `slow_down` to reflect the purpose of the behavior defined in it). Then come the arguments of the function in parenthesis, which also have arbitrary names (in our case there is a single argument that will correspond to the variable representing the agent, so we call it `agent`) and finally the symbol `:`. Below the first line, you find the instructions that this function will execute when it will be called. Those instructions need to be intended right with respect to the first line. In this example, the instructions are the exact same as in the core of the previous `for` loop, except that:
- we omit the last line `sleep(0.1)` (the frequency at which the behavior will be executed will be set in more rigorous way below),
- we don't directly set the motor activations using `agent.left_motor` and `agent.right_motor`. Instead, we *return* the values of the motor activations in the last line and they will be automatically sent to the agent motors when the behavior will be executed. In the last line, the values after the `return` keyword have to be the left and right wheel activation (in this order). Both activations have to be between 0 and 1. (In the `slow_down` behavior above, both activations are the same since we don't want the agent to turn).

Note that a function definition, as the one above, does not execute the instructions contained in it, it only defines them so that they can be executed later when the function will be *called*. In our case, we will not explicitly call the function, instead it will be done behind the scene when we will start the behavior on the agent. To actually start the `slow_down` behavior on the agent we execute:


In [32]:
agent.attach_behavior(slow_down)

The line above means: attach the behavior defined in the function `slow_down` to the `agent` and execute it. You should now see the agent executing the exact same behavior as before. 

When we executed `agent.attach_behavior(slow_down)`, the effect is basically the same as when we executed the `for` loop at the start of this session. Using this method has however the following advantages over the previous method using the `for` loop:
- It is more compact to write and it will allow to better structure your code when you will have to deal with multiple behaviors and multiple agents.
- The behavior will run indefinetely until you explicitely stop it (we'll stop it later).
- It is not blocking as the previous method was. This means that you can still use this notebook while the behavior is running on the agent. For example, let's read the proximeter activations while the agent is still executing the `slow_down` behavior:

In [10]:
agent.proximeters()

[0.0, 0.8085551261901855]

Each time you execute the cell above, you should see the proximeter activation changing because the agent is moving. Note that the proximeter values will be zeros whenever the object is not in the field of view of the agent.

When a behavior is attached, it runs indefinitely until you explicitly detach it, as we will explain below. At anytime, you can also check what behaviors are attached to the agent with the following command:

In [11]:
agent.print_behaviors()

Attached behaviors: ['slow_down'], Started behaviors: ['slow_down']


You don't have to worry about the difference between `Attached behaviors` and `Started behaviors`, you just have to know that the behaviors executing in the simulation are the ones listed in the `Started behaviors` list.

You can detach the behavior of an agent by using the `detach_behavior` method. You have to pass the name od the behavior function you want to detach as an argument. For example, to detach the `slow_down` behavior, you have to execute:

In [33]:
agent.detach_behavior(slow_down)

Now the agent doesn't have any behavior attached to it, but it will continue moving using the last motor speeds it had when the behavior was detached. You can set both motor speeds to 0 by executing:

In [34]:
agent.stop_motors()

We can check that no behavior is curently attached to the agent with:

In [35]:
agent.print_behaviors()

No behavior attached


You can also stop the behavior and stop the motors of the agent with a single instruction. To demonstrate this, let's re-attach the behavior again by executing:

In [15]:
agent.attach_behavior(slow_down)

Now we detach the behavior, this time setting the `stop_motors` argument to `True` in the `detach_behavior` function:

In [16]:
agent.detach_behavior(slow_down, stop_motors=True)

The stop_motors argument will automatically execute `agent.stop_motors()` after detaching the behavior. If you don't want the motors to stop when detaching the behavior, you can set stop_motors to False.

By default, the behaviors of the agents will be executed at every time step of the simulation. You can choose a different interval of behavior execution by setting the `interval` argument when attaching the behavior to the agent. For example, to execute the `slow_down` behavior every 10 steps of the simulation, you can execute:

In [17]:
agent.attach_behavior(slow_down, interval=10)

We recommend to not change the default interval unless the simulator becomes slow. This should not be the case for now, but it might be when we learn to program more complex simulations. 

Now let's detach the behavior of the agent and stop its motors before proceeding to the next section. You can do it with the `detach_all_behaviors` method, which simply detaches all attached behaviors (this way we don't have to indicate the behavior name):

In [18]:
agent.detach_all_behaviors(stop_motors=True)

## Implementing the Braitenberg vehicle behaviors

Let's now practice a bit. Remember the Braitenberg Vehicle examples we have seen in [this slide](https://docs.google.com/presentation/d/1s6ibk_ACiJb9CERJ_8L_b4KFu9d04ZG_htUbb_YSYT4/edit#slide=id.g31e1b425a3_0_0) (if we haven't seen it yet, inform a professor before continuing the session). Those vehicles are very similar to the agents in the simulator. 
- A Braintenberg Vehicle is equipped with two sensors that are activated according to the proximity of a source. With the agent, each proximeter sensor returns a value between 0 and 1 that is inversely proportional to the distance from the closest obstacle it perceives (the closer the obstacle, the highest to proximeter activation). Sensor values are accessed with `agent.proximeters()`
- A Braintenberg Vehicle is equipped with two wheels. An agent in the simulator is also equipped with two wheels, whose rotating speeds are controlled through the activation of each motor independently with a value between 0 and 1 (where 1 means maximum speed). E.g. setting the left wheel at full speed is achieved with `agent.left_motor = 1`, while stopping the right wheel is achieved with `agent.right_motor = 0`.
- A behavior associates sensor activations to motor activations. In the Braitenberg Vehicles, this is achieved through connections that are either excitatory (the activity of the sensor increases the activity of the motor it is connected to) or inhibitory (the activity of the sensor decreases the activity of the motor it is connected to). In the agent, we have seen above that we can define a behavior as a function that (1) read the sensor activities (2) perform some computation and (3) use the result of that computation to set the motor speed. 

Therefore, we can implement the various types of vehicle behaviors shown in the slide on our simulated agent. Defining excitatory and inhibitory connections will be done through Step (2) above (*perform some computation*). We have actually already done it with the `slow_down` behavior we have defined above.  

Let's see how to define the `fear` behavior illustrated in [the slide](https://docs.google.com/presentation/d/1s6ibk_ACiJb9CERJ_8L_b4KFu9d04ZG_htUbb_YSYT4/edit#slide=id.g31e1b425a3_0_0) using the method we have seen: 

In [36]:
def fear(agent):
    left, right = agent.proximeters()
    return left, right

That's pretty easy, isn't it? As illustrated in the slide, the `fear` behavior simply consists in the left sensor exciting the left motor, and the right sensor exciting the right motor. Therefore, the simplest way of programming this behavior is to directly map the left and right sensor activations to the left and right motor speed, respectively. This is what is done in the function definition just above. Since both sensor and motor values are bounded between 0 and 1, there is nothing else to take care of.

Let's now analyze the properties of this `fear` behavior in more detail. Attach and start the `fear` behavior by executing the cell below, and observe how the agent behaves.

In [37]:
agent.detach_all_behaviors()  # Just in case a behavior is still attached
agent.attach_behavior(fear)

Note that this behavior will make the robot move only if at least one if its sensor detects an object. Let's actually add more objects in the scene with:

In [38]:
for obj in controller.objects:
    obj.exists = True

The cell above set the `exists` attribute of all objects to `True`, meaning that all objects are now present in the scene. You should now see 8 objects (the yellow circles) in total.

If no object is within the agent's field of view, you can also drag and drop the agent in a location where it is surrounded by objects. As seen in Session 1, to drag and drop you have to: first click on the `Start Drag & Drop` button above the map, then drag and drop the agent close to an object. The button now displays `Stop Drag & Drop`. Click again on it to complete the operation. Do not forget to click the button again once you have completed your drag and drop, otherwise the simulator will not behave as expected.

**Q1:** Use the cell below to answer the following questions.
1. What happens when the activity of both sensors is null? (i.e. no object is detected.) Why?
2. How does the agent react when it detects an object? Why?
3. Imagine a small animal equipped with such a behavior in the wild. What would be its evolutionary advantages and drawbacks? (could it escape from a predator? could it collect food? Could it hide itself?)

1. agent stops, because left and right sensors are not detecting anything (both 0) so both motors will be 0
2. it "avoids" object, because when the sensor on one side percieves object, motor on that side is activated more, leading agent to turn sideway to object
3. at the moment it can escape a predator partially, it should also continue to "run away" and not stop once it doesn't detect object anymore (continuing in last direction might be beneficial, or randomly changing direction based on last direction (like "zig-zag" but still in more or less the same direction)
it can't collect food, but it might avoid poisonos food :) (maybe if perception of color is also possible to differ good vs bad food)
seems like with "fear" behaviour it could just partially try to escape predators, but hiding probably requires additional/different behaviour
hiding might be artificially constructed by making configuration of objects that lead agents to avoid sequence of them, but that is not far from settings in the wild

**Q2:** Program the `aggression` behavior illustrated in [the slide](https://docs.google.com/presentation/d/1s6ibk_ACiJb9CERJ_8L_b4KFu9d04ZG_htUbb_YSYT4/edit#slide=id.g31e1b425a3_0_0), which consists of crossed excitatory connections. 

In [39]:
def aggression(agent):
    left, right = agent.proximeters()
    return right, left

Before executing the behavior you have defined in the cell just above, first detach the previous one and immobilize the agent:

In [23]:
agent.detach_all_behaviors(stop_motors=True)

Then attach the `aggression` behavior:

In [40]:
agent.attach_behavior(aggression)

**Q3:** Use the cell below to answer the following questions.

3. How does the agent reacts when it approaches an object?  Why?
2. How does the agent react when close to a moveable object (the orange squares in the scene) Why?
4. Imagine an animal equipped with such a behavior in the wild. What would be its evolutionary advantages and drawbacks? (could it escape from a predator? could it catch preys? Could it hide itself? Could it move things?)

3. agent steers toward object, if object is on the side of agent, agent go to that side, becase opposite motor will be activated more, but once object starts being more in front of the agent, both motors "compete" (beceuse both sensors detect similar proximity)
4. since both sensors and both motors are activated agents go straight to the object pushing it 
5. it could move things and catch preys. in order to hide it probably should stop which is hardly possible with this behaviour (it stops only if there is no objects in front, so it's not aware of predators behind). 

## Summary

Let's summarize the method we have just describe to define, attach and detach a behavior.

In [25]:
# First, detach all the behaviors that might still be attached to the agent
# (it is a good practice to do it each time you want to define a new behavior, or modify an existing one):
agent.detach_all_behaviors(stop_motors=True)

In [49]:
# Define a behavior where the agent progressively slows down when it approaches an obstacle:
def slow_down(agent):
    # Step (1): read the sensor values
    left, right = agent.proximeters()
    
    # Step (2): do some computation
    max_prox_value = max(left, right)
    motor_activation = 1.0 - max_prox_value    
    
    # Step (3): return the motor activations
    return motor_activation, motor_activation

In [50]:
# Attach and start this behavior to the agent, and specify the step interval at which it will be executed
import math
agent.orientation = math.pi/2
agent.x_position = 50
agent.y_position = 50

for i in range(len(controller.objects)):
    if i > 0:
        controller.objects[i].exists = False

controller.objects[0].x_position = 50
controller.objects[0].y_position = 80

agent.attach_behavior(slow_down)

When executing the code above, you should see the behavior being executed on the agent in the simulator. Then, to detach the behavior:

In [55]:
print(agent.left_motor)
print(agent.right_motor)

0.501351
0.501351


In [51]:
agent.detach_behavior(slow_down, stop_motors=True)

An alternative way is to detach all the behaviors running on the agent, this avoids having to specify the name of the behavior (`slown_down` in the cell above):

In [46]:
agent.detach_all_behaviors(stop_motors=True)

That's it for this practical session. You can now close Vivarium and deliver your notebook on Aula Global (see instructions on how to do it at the end of Session 1). Before quitting Vivarium, make sure you have saved your notebook. 